# Setting up a Simulation

First, we import the modules we need:

In [22]:
from ase.io import read, write
from ase.visualize import view

### Choosing the Starting structure

Now we load the structure that we previously created:

In [33]:
atoms = read("03_structure.cif")

Quick check if the structure looks correct:

In [24]:
view(atoms, viewer="x3d")

### Choosing the Calculator

We want to do simulations with MACE so we use the MACE

In [ ]:
from mace.calculators import mace

### Choosing the Model

Now that we got our initial Structure we can choose a model that suits our use case. There are different models available.

Pick a machine‑learned potential suited to your system:

- Universal MOFs and general solids: https://github.com/ACEsuit/mace-foundations/releases/tag/mace_mp_0b (MACE‑MP‑0b, different models, recommended: 
mace_agnesi_small.model)
- Newest MACE model for materials: https://github.com/ACEsuit/mace-foundations (MACE-mh-1, head: omat_pbe)
- MOF‑specific: https://github.com/ddmms/data/tree/main/mace-mof-0/v2 (MACE‑MP‑MOFv2, head: pbe_d3)
- Small molecules: github.com/ACEsuit/mace-off (MACE‑OFF; not suited for MOFs)
- Alternative approach:  (UMA different methodology)


General: MACE Foundations (models and docs): https://github.com/ACEsuit/mace-foundations

For this workshop we want to use a MACE model that suits MOFs (mace_agnesi_small.model, mace-mh-1.model or mofs_v2.model).

Now we include the model. Adjust the path to where you saved the model you downloaded

The models are float64 by default but we use float32 to conserve computational effort.

When we calculate our trajectories we use CUDA but for now to check we can simply use the cpu.

In [36]:
model = "../../models/mace-mh-1.model"
default_dtype = "float32"
calc = mace.MACECalculator(model_paths=model, default_dtype=default_dtype ,device="cpu", head="omat_pbe")

/scratch/data/marion/miniforge3/envs/ml/lib/python3.13/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [37]:
atoms.calc = calc

# Test energy/forces
e = atoms.get_potential_energy()
f = atoms.get_forces()
print("Initial energy (eV):", e)
print("Max force (eV/Å):", abs(f).max())

Initial energy (eV): -862.9823608398438
Max force (eV/Å): 4.1602993


### Minimizing the system

we can minimize the system with different optimizers. The

In [40]:
from ase.optimize import BFGS

In [41]:
opt = BFGS(atoms, logfile=None)
opt.run(fmax=0.05)  # relax until max force <= 0.05 eV/Å

np.True_

In [42]:
view(atoms, viewer="x3d")

we can save the minimized structure:

In [43]:
write("03_opt_struct.cif", atoms)

### Equilibrate the system

In [44]:
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.io.trajectory import Trajectory
from ase import units
import numpy as np
import os

We initialize the velocities using a Maxwell-Boltzmann-distribution and remove the net momentum.

In [ ]:
MaxwellBoltzmannDistribution(atoms, temperature_K=300.0, rng=np.random.default_rng(42))
Stationary(atoms)

For equilibration we use a small timestep of 0.25 femtoseconds. We can use different types of thermostats. We are going to use Langevin dynamics. 

We make sure that we create all the directories we need.

In [ ]:
dt = 0.25 * units.fs
fric = 0.05
dyn = Langevin(atoms, dt, temperature_K=300.0, friction=fric)

report_interval = 100

steps = 1000

# 2) Clean observers, attach logger once
dyn.observers = []
os.makedirs("traj", exist_ok=True)
os.makedirs("out", exist_ok=True)

Now that we set all the parameters we can run the simulation.

In [ ]:
with Trajectory("traj/03_equilibration.traj", "w", atoms) as traj:
    dyn.attach(traj.write, interval=report_interval)

    def print_status():
        ekin = atoms.get_kinetic_energy()
        epot = atoms.get_potential_energy()
        N = len(atoms)
        T = 2.0 * ekin / (3.0 * N * units.kB)  # adjust ndof if you have constraints
        print(f"step={dyn.get_number_of_steps():4d}  Epot={epot:10.3f} eV  "
              f"Ekin={ekin:10.3f} eV  T={T:7.1f} K")
    dyn.attach(print_status, interval=report_interval)

    dyn.run(steps)

### Running the equilibraion on the cluster

This equilibration was just a short example on how we run the simulation. When we simulate real trajectories we need more steps and therefore do not calculate on our local computer but we send them to the cluster. Therefore, we need to wrap our code up in a python script which is already provided in 03_equilibration.py. It contains the same information that we need here but uses CUDA and we do the equilibration for 100000 steps instead of 1000.

To run the simulations on the cluster we also need a bash script 03_equilibration.sh which is already provided (you need to make adjustment depending on the configuration of your cluster)

#### Congratulations we have an equlibrated system!

### Running the simulation on the cluster

When the equilibration is finished, we can run the simulation starting from the last step of our trajectory file. The file 04_nvt.py is the simulation file is nearly identical to the equilibration but we use a larger time step and more total steps. The bash script to start the simulation is also provided as 04_nvt.sh. It is important to save not only coordinates but also velocities if you want to start from your last trajectory frame. There are several fomats ASE I/O can handle. The native one is the .traj format which is very compact and stores all necessary data, however it is not well transferrable to other modules like MDAnalysis. 

Storing your trajectory as a .traj file is always a good start, if you later on realize something else would have been better you can transform the trajectory.

Link to I/O options including possible format: https://docs.ase-lib.org/ase/io/io.html

note that ase uses a special extended xyz format, to save files as simple xyz use:



In [49]:
lastframe = read("traj/03_equilibration.traj", index = "-1")
write("04_equilibrated_structure.xyz", lastframe, format = "xyz")

While we are waiting to finish our calculation which probably takes longer than the workshop, we can take a look at the prepared trajectories. They are stored in the xyz format.

#### Congratulations we have the final trajectoty!